In [ ]:
__author__ = 'Cyrille Mvomo, https://github.com/cyrillemvomo'
__version__ = "1"
__license__ = "MIT"

**Import**

In [ ]:
import sys
sys.path.append("/Users/cyrilleetude/Documents/GitHub/Fatigue_Study_Parkinson/1_Extract")
sys.path.append("/Users/cyrilleetude/Documents/GitHub/Fatigue_Study_Parkinson/2_Transform")
from Extracted import Extracted_Data
import numpy as np
import pandas as pd
import copy #so that I can copy object
import pickle

**Read**

In [ ]:
EXTRACTED_DATA_STRAIGHT = Extracted_Data(Gait_Data_Path = "/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/McGill_Cohort_2_just_wearables/Wearables/Straight", 
                                Medical_Record_Path = "/Volumes/Active/Datasets/Fatigue_Project/Demographic", 
                                Source_Folder_Path = "/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/0_Source").RawData_Extracted(save_data=False)

EXTRACTED_DATA_STEERING = Extracted_Data(Gait_Data_Path = "/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/McGill_Cohort_2_just_wearables/Wearables/Steering", 
                                Medical_Record_Path = "/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/McGill_Cohort_2_just_wearables/Demographic", 
                                Source_Folder_Path = "/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/0_Source").RawData_Extracted(save_data=False)

**Crop**

In [ ]:
# Declare variables
sampling_rate= 128 #hz
duration_start_walking = 120 + 10 #number of seconds between injection of FDG (start of APDM data collection) and start of gait task + 10 second to remove start effect
gait_start = duration_start_walking * sampling_rate# convert in frames
gait_end = gait_start + (120*sampling_rate) # stop after 2 min (in frames)

In [ ]:
def Crop(ID, medicalrecord_df, df_straight, df_steering, to_crop_straight_start = [], to_crop_steering_start = []
         , to_crop_straight_10 = [], to_crop_steering_10 = [], to_crop_straight_20 = [], to_crop_steering_20 = []):
    '''
    Function to crop the signal
        input:
        - ID : string representing the Participant ID
        - medicalrecord_df : df containing medical record info
        - df_straight : data frame containing acceleration signal in straight condition
        - df_steering : data frame containing acceleration signal in steering condition
        - to_crop_straight_start : list of 2 integers corresponding frames to crop in the first 2 min of gait (straight condition) (default = [])
        - to_crop_straight_10 : list of 2 integers corresponding frames to crop in the first 2 min after 10 min of gait (straight condition) (default = [])
        - to_crop_straight_20 : list of 2 integers corresponding frames to crop in the first 2 min after 20 min of gait (straight condition) (default = [])
        - to_crop_steering_start : list of 2 integers corresponding frames to crop in the first 2 min of gait (steering condition) (default = [])
        - to_crop_steering_10 : list of 2 integers corresponding frames to crop in the first 2 min after 10 min of gait (steering condition) (default = [])
        - to_crop_steering_20 : list of 2 integers corresponding frames to crop in the first 2 min after 20 min of gait (steering condition) (default = [])

        output: a dict with 6 dfs
            - cropped_straight_start : df containing the cropped signal
            - cropped_straight_10min : df containing the cropped signal
            - cropped_straight_20min : df containing the cropped signal
            - cropped_steering_start : df containing the cropped signal
            - cropped_steering_10min : df containing the cropped signal
            - cropped_steering_20min : df containing the cropped signal
    '''
    # Declare
    sampling_rate= 128 #hz used to convert in frames
    gait_start_straight = (medicalrecord_df.loc[medicalrecord_df.index == ID]["START_S_APDM_STRAIGHT"].values[0] + 30) #apdm s when gait task start  + 30 seconds (start effects)
    gait_10min_straight = gait_start_straight + 600 # after 10 min
    gait_20min_straight = gait_10min_straight + (600) # after 20 min
    # acceleration_vertical_straight = df_straight['ACC_VERTICAL_MS2'].values[gait_start_straight*sampling_rate:]
    # time_straight = df_straight['TIME_S'].values[gait_start_straight*sampling_rate:]
    gait_start_steering = (medicalrecord_df.loc[medicalrecord_df.index == ID]["START_S_APDM_STEERING"].values[0] + 30) #apdm s when gait task start + 30 seconds (start effects)
    gait_10min_steering  = gait_start_steering  + 600 # after 10 min
    gait_20min_steering  = gait_10min_steering  + (600) # after 20 min
    # acceleration_vertical_steering  = df_steering['ACC_VERTICAL_MS2'].values[gait_start_steering*sampling_rate:]
    # time_steering = df_steering ['TIME_S'].values[gait_start_steering*sampling_rate:]

    # Crop
        # Straight
    if len(to_crop_straight_start) == 0: # if no specified comments marked
        to_crop_straight_start = [gait_start_straight*sampling_rate, (gait_start_straight+120)*sampling_rate] # crop at frame start + 30 s (start effect) and frame 2 min later
    else:
        to_crop_straight_start = to_crop_straight_start #specified frames
        
    if len(to_crop_straight_10) == 0: # if no specified comments marked
        to_crop_straight_10 = [gait_10min_straight*sampling_rate, (gait_10min_straight+120)*sampling_rate] # crop at frame 10 min and frame 2 min later
    else:
        to_crop_straight_10 = to_crop_straight_10 #specified frames
        
    if len(to_crop_straight_20) == 0: # if no specified comments marked
        to_crop_straight_20 = [gait_20min_straight*sampling_rate, (gait_20min_straight+120)*sampling_rate] # crop at frame 20 min and frame 2 min later
    else:
        to_crop_straight_20 = to_crop_straight_20 #specified frames

        # Steering
    if len(to_crop_steering_start) == 0: # if no specified comments marked
        to_crop_steering_start = [gait_start_steering*sampling_rate, (gait_start_steering+120)*sampling_rate] # crop at frame start + 30 s (start effect) and frame 2 min later
    else:
        to_crop_steering_start = to_crop_steering_start #specified frames
        
    if len(to_crop_steering_10) == 0: # if no specified comments marked
        to_crop_steering_10 = [gait_10min_steering*sampling_rate, (gait_10min_steering+120)*sampling_rate] # crop at frame 10 min and frame 2 min later
    else:
        to_crop_steering_10 = to_crop_steering_10 #specified frames
        
    if len(to_crop_steering_20) == 0: # if no specified comments marked
        to_crop_steering_20 = [gait_20min_steering*sampling_rate, (gait_20min_steering+120)*sampling_rate] # crop at frame 20 min and frame 2 min later
    else:
        to_crop_steering_20 = to_crop_steering_20 #specified frames

    return {
            "cropped_straight_start" : df_straight.iloc[to_crop_straight_start[0]:to_crop_straight_start[1], :],
            "cropped_straight_10min" : df_straight.iloc[to_crop_straight_10[0]:to_crop_straight_10[1], :],
            "cropped_straight_20min" : df_straight.iloc[to_crop_straight_20[0]:to_crop_straight_20[1], :],
            "cropped_steering_start" : df_steering.iloc[to_crop_steering_start[0]:to_crop_steering_start[1], :],
            "cropped_steering_10min" : df_steering.iloc[to_crop_steering_10[0]:to_crop_steering_10[1], :],
            "cropped_steering_20min" : df_steering.iloc[to_crop_steering_20[0]:to_crop_steering_20[1], :]
            }

In [ ]:
id = "PD" + "40"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : clean \n\n",
      "steering : clean ")

In [ ]:
id = "PD" + "61"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [128*1700, 128*1800], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : clean \n\n",
      "steering : for 10 min, use from 1700 as pb at 1690 s ")

In [ ]:
id = "PD" + "66"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : clean \n\n",
      "steering : clean ")

In [ ]:
id = "PD" + "94"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
# Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
# Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
# Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
# Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : clean \n\n",
      "steering : clean ")

In [ ]:
id = "PD" + "80"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : clean \n\n",
      "steering : clean ")

In [ ]:
id = "PD" + "81"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [128*1150, 128*1180], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [128*1750, 128*1850], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [128*2340, 128*2360])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : clean \n\n",
      "steering : take only 30 second (after start) and for the beggining,  for 10 min start at 1750, for 20 min take up to 20 seconds after start (stopped earlier)")

In [ ]:
id = "PD" + "82"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : clean \n\n",
      "steering : clean")

In [ ]:
id = "PD" + "83"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [128*1210, 128*1240], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
# Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
# Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
# Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : to review \n\n",
      "steering : beggining take 30 second")

In [ ]:
id = "PD" + "84"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : clean \n\n",
      "steering : clean")

In [ ]:
id = "PD" + "85"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : clean \n\n",
      "steering : clean")

In [ ]:
id = "PD" + "87"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [], 
          to_crop_straight_20 = [128* 1740, 128*1760], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : for the 20 min, take the last 20 seconds \n\n",
      "steering : clean")

In [ ]:
id = "PD" + "88"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [660*128, 750*128], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : clean \n\n",
      "steering : second start at 660")

In [ ]:
id = "PD" + "89"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [1870*128, 1900*128])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
# Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : drop last \n\n",
      "steering : last 20 min only first 30 s")

In [ ]:
id = "PD" + "90"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [1430*128, 1480*128], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
# Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
# Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
# Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : Only take the first 2 min (to noisy FOG ++++) \n\n",
      "steering : Second 10 min start at 1430 and discard the last 2 min at 20 min")

In [ ]:
id = "PD" + "91"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
# Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : for the last 20 min discard \n\n",
      "steering : clean")

In [ ]:
id = "PD" + "92"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [850*128, 900*128], 
          to_crop_straight_20 = [1524*128, 1554*128], 
          to_crop_steering_20 = [1500*128, 1550*128])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : for the last 20 min take from 1524 to 1554s \n\n",
      "steering : second bout up to 900 and last bout from 1500 onward")

In [ ]:
id = "PD" + "93"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [830*128, 890*128], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : clean \n\n",
      "steering : second bout from 830 to 890")

In [ ]:
id = "PD" + "95"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [2300*128, 2350*128], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [2930*128, 2960*128])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : clean \n\n",
      "steering : second start at 2300 last = last 30 seconds")

In [ ]:
id = "PD" + "96"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : clean \n\n",
      "steering : clean")

In [ ]:
id = "PD" + "98"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [1600*128, 1630*128], 
          to_crop_steering_start = [380*128, 410*128], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [], 
          to_crop_straight_20 = [2790*128, 2820*128], 
          to_crop_steering_20 = [1550*128, 1600*128])

# Save as excel file based on comments
Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : first bout from 1600 to 1630 last from 2790 to the end \n\n",
      "steering : first bout 380-410, last 1550-1600")

In [ ]:
id = "PD" + "99"
Cropped = Crop(ID = id, 
          medicalrecord_df = EXTRACTED_DATA_STRAIGHT["Medical_Record"], 
          df_straight = EXTRACTED_DATA_STRAIGHT[id], 
          df_steering = EXTRACTED_DATA_STEERING[id], 
          to_crop_straight_start = [], 
          to_crop_steering_start = [], 
          to_crop_straight_10 = [], 
          to_crop_steering_10 = [], 
          to_crop_straight_20 = [], 
          to_crop_steering_20 = [])

# Save as excel file based on comments
# Cropped["cropped_straight_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/Start/{id}.csv", index = False)
# Cropped["cropped_straight_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/10min/{id}.csv", index = False)
# Cropped["cropped_straight_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Straight/20min/{id}.csv", index = False)
Cropped["cropped_steering_start"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/Start/{id}.csv", index = False)
Cropped["cropped_steering_10min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/10min/{id}.csv", index = False)
Cropped["cropped_steering_20min"].to_csv(f"/Users/cyrilleetude/Documents/GitHub/PD_Gait_Wearables_NeuroImaging_Biomarkers/Data_Analysis/Preprocessing/MoCap/McGill_Cohort_2_without_imaging/2_Transform/0_Signal_Processing/Script_Bella/Cropped/Steering/20min/{id}.csv", index = False)

#comment
print("straight : to review (lumbar apdm issue otherwise good) \n\n",
      "steering : clean")